# Your First LLM-Powered Tool

In this lab, I build a small LLM-powered support ticket classifier.

The classifier receives a short customer message and predicts one of four labels:

- `billing`
- `bug`
- `feature_request`
- `other`

I use Gemini as the hosted LLM and Ollama with `llama3.2:3b` as the local fallback model.

In [1]:
# Setup
import os
import json
import re
import urllib.request
import urllib.error
from pathlib import Path

from google import genai
from google.genai import types

In [2]:
def load_env_file(path=".env"):
    """
    Load simple KEY=VALUE pairs from a local .env file into os.environ.
    This avoids writing secrets directly inside the notebook.
    """
    env_path = Path(path)

    if not env_path.exists():
        return

    with env_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("#") or "=" not in line:
                continue

            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")

            if key and value and key not in os.environ:
                os.environ[key] = value


load_env_file()

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY in your environment or local .env file."

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.0-flash"

print("Gemini API key loaded:", bool(GEMINI_API_KEY))
print("Model:", MODEL)

Gemini API key loaded: True
Model: gemini-2.0-flash


## Task 1 — First calls and the sampling dial

In this task, I make a working Gemini API call with a system instruction and a user message.  
Then I run the same prompt multiple times with low and high temperature values to observe how sampling changes the output.

**Note:** My Gemini API key was loaded successfully, but the Gemini request returned a `429 RESOURCE_EXHAUSTED` quota error. Because the lab allows Ollama as a fallback, I continue the lab using the local `llama3.2:3b` model through Ollama.

In [3]:
OLLAMA_MODEL = "llama3.2:3b"
OLLAMA_URL = "http://localhost:11434/v1/chat/completions"


def print_usage_from_dict(usage):
    """
    Print token usage from an OpenAI-compatible usage dictionary.
    """
    if not usage:
        print("Usage metadata: not available")
        return

    print("Usage metadata:")
    print("Prompt tokens:", usage.get("prompt_tokens"))
    print("Output tokens:", usage.get("completion_tokens"))
    print("Total tokens:", usage.get("total_tokens"))


def ollama_generate(prompt, system_instruction=None, temperature=0.2, max_output_tokens=256):
    """
    Send a prompt to local Ollama using its OpenAI-compatible endpoint.
    """
    messages = []

    if system_instruction:
        messages.append({
            "role": "system",
            "content": system_instruction
        })

    messages.append({
        "role": "user",
        "content": prompt
    })

    payload = {
        "model": OLLAMA_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_output_tokens,
        "stream": False,
    }

    data = json.dumps(payload).encode("utf-8")

    request = urllib.request.Request(
        OLLAMA_URL,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            response_data = json.loads(response.read().decode("utf-8"))

        text = response_data["choices"][0]["message"]["content"]
        usage = response_data.get("usage")

        return {
            "provider": "ollama",
            "text": text,
            "usage": usage,
            "raw": response_data,
        }

    except Exception as e:
        return {
            "provider": "ollama",
            "text": f"Ollama call failed: {type(e).__name__}: {e}",
            "usage": None,
            "raw": None,
        }

In [4]:
system_message = "You are a concise support assistant."

user_message = """
A customer says: "I was charged twice for my subscription this month."
Reply in one concise sentence explaining what the support team should do next.
"""

response = ollama_generate(
    prompt=user_message,
    system_instruction=system_message,
    temperature=0.2,
)

print("Provider:", response["provider"])
print()
print("Response:")
print(response["text"])
print()
print_usage_from_dict(response["usage"])

Provider: ollama

Response:
The support team should investigate the duplicate charge by checking the customer's account history and contact the billing department to verify and correct the error.

Usage metadata:
Prompt tokens: 61
Output tokens: 28
Total tokens: 89


In [5]:
sampling_prompt = """
Classify this customer support ticket as one of:
billing, bug, feature_request, or other.

Ticket:
"The export button throws a 500 error every time I click it on the reports page."

Return a short answer with the label and one brief reason.
"""

for temperature in [0.1, 1.0]:
    print(f"\n==============================")
    print(f"Temperature: {temperature}")
    print(f"==============================")

    for run in range(1, 4):
        response = ollama_generate(
            prompt=sampling_prompt,
            system_instruction="You are a concise support ticket classifier.",
            temperature=temperature,
            max_output_tokens=120,
        )

        print(f"\nRun {run}")
        print("Provider:", response["provider"])
        print(response["text"])


Temperature: 0.1

Run 1
Provider: ollama
Feature Request
The issue is related to functionality, specifically an error when using a specific feature.

Run 2
Provider: ollama
Feature Request
The issue is related to functionality, specifically an error message when using a specific feature.

Run 3
Provider: ollama
Feature Request
The issue is related to functionality, specifically an error when using a specific feature.

Temperature: 1.0

Run 1
Provider: ollama
Label: Bug
Reason: Error in critical functionality (export button) resulting in a visible issue.

Run 2
Provider: ollama
**bug**

Error on report page.

Run 3
Provider: ollama
Label: Bug
Reason: Error occurs when attempting to use interactive feature (export button).


### What changed, and why?

Gemini was attempted first, but the API returned a `429 RESOURCE_EXHAUSTED` quota error. Because the lab allows a local fallback, I continued the lab using Ollama with the `llama3.2:3b` model.

The first Ollama call worked successfully and returned token usage: 61 prompt tokens, 28 output tokens, and 89 total tokens.

For the sampling experiment, the low-temperature runs (`0.1`) were more consistent in wording and all three outputs predicted `feature_request`. However, this was not the best label because a `500 error` usually indicates a technical bug. At high temperature (`1.0`), the outputs varied more in wording, but all three runs predicted `bug`, which is the better label for this ticket.

This shows that lower temperature usually makes outputs more stable, but it does not guarantee correctness. For classification tasks, prompt quality and examples are still important, not only the sampling temperature.

## Task 2 — Prompt-pattern bake-off

In this task, I classify the same support tickets using three prompt patterns:

1. Zero-shot prompting
2. Few-shot prompting
3. Decomposition prompting

I compare the predicted labels in a table to see which prompt pattern is more reliable.

In [6]:
with open("tickets.json", "r", encoding="utf-8") as f:
    tickets = json.load(f)

LABELS = ["billing", "bug", "feature_request", "other"]

print("Number of tickets:", len(tickets))

for ticket in tickets:
    print(ticket["id"], "-", ticket["text"])

Number of tickets: 8
1 - I was charged twice for my subscription this month. Please refund the extra charge.
2 - The export button throws a 500 error every time I click it on the reports page.
3 - It would be great if you could add a dark mode to the dashboard.
4 - How do I reset my password? I can't find the link anywhere.
5 - The app crashes on startup after the latest update on Android 14.
6 - Can you send me a copy of my last invoice for our accounting team?
7 - Please add PDF export — CSV alone isn't enough for our reports.
8 - Just wanted to say the new UI looks really clean. Nice work!


In [8]:
ZERO_SHOT_PROMPT = """
Classify the customer support ticket into exactly one of these labels:

- billing
- bug
- feature_request
- other

Return only the label.

Ticket:
{text}
"""


FEW_SHOT_PROMPT = """
Classify the customer support ticket into exactly one of these labels:

- billing
- bug
- feature_request
- other

Examples:

Ticket: I was charged twice for my subscription this month.
Label: billing

Ticket: The app crashes whenever I open it after the latest update.
Label: bug

Ticket: Please add dark mode to the dashboard.
Label: feature_request

Ticket: Just wanted to say the new UI looks really clean.
Label: other

Return only the label.

Ticket:
{text}

Label:
"""


DECOMPOSITION_PROMPT = """
Classify the customer support ticket into exactly one of these labels:

- billing
- bug
- feature_request
- other

Briefly identify the main signal in the ticket, then give the final label.

Use this exact format:
Signal: short phrase
Label: one label only

Ticket:
{text}
"""

In [9]:
def extract_label(raw_text):
    """
    Extract one valid label from the model output.
    """
    if raw_text is None:
        return "invalid"

    text = raw_text.strip().lower()

    for label in LABELS:
        pattern = r"\b" + re.escape(label) + r"\b"
        if re.search(pattern, text):
            return label

    return "invalid"

In [10]:
def build_prompt(text, style):
    if style == "zero_shot":
        return ZERO_SHOT_PROMPT.format(text=text)

    if style == "few_shot":
        return FEW_SHOT_PROMPT.format(text=text)

    if style == "decomposition":
        return DECOMPOSITION_PROMPT.format(text=text)

    raise ValueError(f"Unknown style: {style}")


def classify(text, style):
    prompt = build_prompt(text, style)

    response = ollama_generate(
        prompt=prompt,
        system_instruction="You are a careful support ticket classifier.",
        temperature=0.1,
        max_output_tokens=120,
    )

    raw_output = response["text"]
    label = extract_label(raw_output)

    return label, raw_output

In [11]:
styles = ["zero_shot", "few_shot", "decomposition"]
results = []

for ticket in tickets:
    row = {
        "id": ticket["id"],
        "text": ticket["text"],
    }

    for style in styles:
        label, raw_output = classify(ticket["text"], style)
        row[style] = label
        row[f"{style}_raw"] = raw_output.strip()

    results.append(row)

print("Bake-off completed.")

Bake-off completed.


In [12]:
print("| ID | Ticket | Zero-shot | Few-shot | Decomposition |")
print("|---:|---|---|---|---|")

for row in results:
    ticket_text = row["text"].replace("|", "\\|")

    print(
        f"| {row['id']} | {ticket_text} | "
        f"{row['zero_shot']} | {row['few_shot']} | {row['decomposition']} |"
    )

| ID | Ticket | Zero-shot | Few-shot | Decomposition |
|---:|---|---|---|---|
| 1 | I was charged twice for my subscription this month. Please refund the extra charge. | billing | billing | billing |
| 2 | The export button throws a 500 error every time I click it on the reports page. | bug | bug | bug |
| 3 | It would be great if you could add a dark mode to the dashboard. | feature_request | feature_request | feature_request |
| 4 | How do I reset my password? I can't find the link anywhere. | feature_request | other | bug |
| 5 | The app crashes on startup after the latest update on Android 14. | bug | bug | bug |
| 6 | Can you send me a copy of my last invoice for our accounting team? | billing | billing | billing |
| 7 | Please add PDF export — CSV alone isn't enough for our reports. | feature_request | feature_request | feature_request |
| 8 | Just wanted to say the new UI looks really clean. Nice work! | other | other | feature_request |


In [13]:
for row in results:
    print("=" * 80)
    print(f"Ticket {row['id']}: {row['text']}")
    print()

    print("Zero-shot raw output:")
    print(row["zero_shot_raw"])
    print()

    print("Few-shot raw output:")
    print(row["few_shot_raw"])
    print()

    print("Decomposition raw output:")
    print(row["decomposition_raw"])
    print()

Ticket 1: I was charged twice for my subscription this month. Please refund the extra charge.

Zero-shot raw output:
billing

Few-shot raw output:
billing

Decomposition raw output:
Signal: duplicate charge
Label: billing

Ticket 2: The export button throws a 500 error every time I click it on the reports page.

Zero-shot raw output:
bug

Few-shot raw output:
bug

Decomposition raw output:
Signal: Error with export button
Label: bug

Ticket 3: It would be great if you could add a dark mode to the dashboard.

Zero-shot raw output:
feature_request

Few-shot raw output:
feature_request

Decomposition raw output:
Signal: feature request
Label: feature_request

Ticket 4: How do I reset my password? I can't find the link anywhere.

Zero-shot raw output:
feature_request

Few-shot raw output:
other

Decomposition raw output:
Signal: password reset issue
Label: bug

Ticket 5: The app crashes on startup after the latest update on Android 14.

Zero-shot raw output:
bug

Few-shot raw output:
bug



### Task 2 observations

The few-shot prompt was the most reliable pattern in this bake-off. It correctly handled clear billing, bug, and feature request tickets, and it also classified the password reset question as `other`, which was the best label among the available categories.

The zero-shot prompt worked well for most obvious tickets, but it failed on the password reset ticket by labeling it as `feature_request`. This shows that without examples, the model may confuse a general support question with a request for a new feature.

The decomposition prompt was useful because it gave a visible signal before the final label, but it was less reliable in this run. It mislabeled the password reset ticket as `bug` and the UI compliment as `feature_request`.

Overall, few-shot prompting won because the examples helped the model understand the intended boundaries between `billing`, `bug`, `feature_request`, and `other`.

## Task 3 — Structured output as a function

In this task, I rewrite the classifier so it returns structured JSON instead of free text.

The required schema is:

```json
{
  "label": "billing | bug | feature_request | other",
  "confidence": 0.0,
  "reason": "short string"
}
```

Because Gemini returned a quota error earlier, I continue this structured-output task with the local Ollama `llama3.2:3b` model. I still validate the output in Python instead of trusting the model blindly.

In [30]:
STRUCTURED_PROMPT = """
Classify the following customer support ticket.

Allowed labels and definitions:
- billing: payments, charges, refunds, invoices, subscriptions, receipts, or accounting requests
- bug: crashes, errors, broken features, failed actions, or unexpected technical behavior
- feature_request: requests to add, improve, or change product functionality
- other: general questions, compliments, password help, or messages that do not fit the other categories

Return only valid JSON with these fields:
- label: one of billing, bug, feature_request, other
- confidence: a number between 0 and 1 showing how confident you are
- reason: a short specific reason based on the ticket text

Do not copy placeholder values.
Do not write "short string" as the reason.
Do not include markdown, explanations, or extra text outside the JSON.

Example output:
{{
  "label": "bug",
  "confidence": 0.87,
  "reason": "The customer reports a 500 error when using the export button."
}}

Ticket:
{text}
"""

def validate_classification(data):
    """
    Validate parsed classification output.
    """
    if not isinstance(data, dict):
        return False, "Output is not a JSON object."

    required_keys = ["label", "confidence", "reason"]

    for key in required_keys:
        if key not in data:
            return False, f"Missing required key: {key}"

    if data["label"] not in LABELS:
        return False, f"Invalid label: {data['label']}"

    if not isinstance(data["confidence"], (int, float)):
        return False, "Confidence must be a number."

    if not 0 <= data["confidence"] <= 1:
        return False, "Confidence must be between 0 and 1."

    if not isinstance(data["reason"], str) or not data["reason"].strip():
        return False, "Reason must be a non-empty string."

    placeholder_reasons = [
        "short string",
        "longer string explaining the request",
        "reason",
        "string"
    ]

    if data["reason"].strip().lower() in placeholder_reasons:
        return False, "Reason looks like a copied placeholder."

    return True, "Valid output."



def parse_and_validate_json(raw_text):
    """
    Parse JSON safely and validate it.
    """
    try:
        data = json.loads(raw_text)
    except json.JSONDecodeError as e:
        return None, False, f"Invalid JSON: {e}"

    is_valid, message = validate_classification(data)
    return data, is_valid, message

In [31]:
def ollama_generate_json(prompt, system_instruction=None, temperature=0.1, max_output_tokens=200):
    """
    Send a prompt to local Ollama and ask for JSON output.
    Uses Ollama's OpenAI-compatible endpoint.
    """
    messages = []

    if system_instruction:
        messages.append({
            "role": "system",
            "content": system_instruction
        })

    messages.append({
        "role": "user",
        "content": prompt
    })

    payload = {
        "model": OLLAMA_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_output_tokens,
        "stream": False,
        "response_format": {"type": "json_object"},
    }

    data = json.dumps(payload).encode("utf-8")

    request = urllib.request.Request(
        OLLAMA_URL,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            response_data = json.loads(response.read().decode("utf-8"))

        text = response_data["choices"][0]["message"]["content"]

        return {
            "provider": "ollama",
            "text": text,
            "usage": response_data.get("usage"),
            "raw": response_data,
        }

    except Exception as e:
        return {
            "provider": "ollama",
            "text": f"Ollama JSON call failed: {type(e).__name__}: {e}",
            "usage": None,
            "raw": None,
        }

In [32]:
def classify_structured(text):
    """
    Classify a support ticket and return parsed + validated JSON output.
    """
    prompt = STRUCTURED_PROMPT.format(text=text)

    response = ollama_generate_json(
        prompt=prompt,
        system_instruction="You are a careful support ticket classifier. Return only valid JSON.",
        temperature=0.1,
        max_output_tokens=200,
    )

    raw_text = response["text"]
    data, is_valid, message = parse_and_validate_json(raw_text)

    return {
        "provider": response["provider"],
        "raw": raw_text,
        "data": data,
        "valid": is_valid,
        "message": message,
        "usage": response["usage"],
    }

In [33]:
structured_results = []

for ticket in tickets:
    result = classify_structured(ticket["text"])

    structured_results.append({
        "id": ticket["id"],
        "text": ticket["text"],
        "label": result["data"]["label"] if result["data"] else None,
        "confidence": result["data"]["confidence"] if result["data"] else None,
        "reason": result["data"]["reason"] if result["data"] else None,
        "valid": result["valid"],
        "message": result["message"],
        "raw": result["raw"],
    })

print("| ID | Label | Confidence | Valid | Reason |")
print("|---:|---|---:|---|---|")

for row in structured_results:
    reason = str(row["reason"]).replace("|", "\\|")
    print(
        f"| {row['id']} | {row['label']} | {row['confidence']} | "
        f"{row['valid']} | {reason} |"
    )

| ID | Label | Confidence | Valid | Reason |
|---:|---|---:|---|---|
| 1 | billing | 1 | True | Customer requests refund for duplicate charge |
| 2 | bug | 0.93 | True | The customer reports a 500 error when using the export button. |
| 3 | feature_request | 1 | True | Request for product feature: dark mode |
| 4 | other | 0.83 | True | Customer is looking for password reset instructions |
| 5 | bug | 0.83 | True | app crashes on startup |
| 6 | other | 0.83 | True | Request for invoice copy |
| 7 | feature_request | 1 | True | Customer requests to add PDF export feature |
| 8 | other | 0.95 | True | Compliment on product design |


In [34]:
for row in structured_results:
    print("=" * 80)
    print(f"Ticket {row['id']}: {row['text']}")
    print("Valid:", row["valid"])
    print("Validation message:", row["message"])
    print("Raw output:")
    print(row["raw"])
    print()

Ticket 1: I was charged twice for my subscription this month. Please refund the extra charge.
Valid: True
Validation message: Valid output.
Raw output:
{
  "label": "billing",
  "confidence": 1,
  "reason": "Customer requests refund for duplicate charge"
}

Ticket 2: The export button throws a 500 error every time I click it on the reports page.
Valid: True
Validation message: Valid output.
Raw output:
{
  "label": "bug",
  "confidence": 0.93,
  "reason": "The customer reports a 500 error when using the export button."
}

Ticket 3: It would be great if you could add a dark mode to the dashboard.
Valid: True
Validation message: Valid output.
Raw output:
{
  "label": "feature_request",
  "confidence": 1,
  "reason": "Request for product feature: dark mode"
}

Ticket 4: How do I reset my password? I can't find the link anywhere.
Valid: True
Validation message: Valid output.
Raw output:
{
  "label": "other",
  "confidence": 0.83,
  "reason": "Customer is looking for password reset instruct

In [35]:
bad_outputs = [
    "billing",
    "{label: billing, confidence: high}",
    json.dumps({
        "label": "payment",
        "confidence": 0.8,
        "reason": "Mentions payment."
    }),
    json.dumps({
        "label": "billing",
        "confidence": 1.4,
        "reason": "Mentions invoice."
    }),
    json.dumps({
        "label": "bug",
        "confidence": 0.7
    }),
]

for bad in bad_outputs:
    data, is_valid, message = parse_and_validate_json(bad)

    print("=" * 80)
    print("Raw bad output:")
    print(bad)
    print("Parsed data:", data)
    print("Valid:", is_valid)
    print("Message:", message)

Raw bad output:
billing
Parsed data: None
Valid: False
Message: Invalid JSON: Expecting value: line 1 column 1 (char 0)
Raw bad output:
{label: billing, confidence: high}
Parsed data: None
Valid: False
Message: Invalid JSON: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Raw bad output:
{"label": "payment", "confidence": 0.8, "reason": "Mentions payment."}
Parsed data: {'label': 'payment', 'confidence': 0.8, 'reason': 'Mentions payment.'}
Valid: False
Message: Invalid label: payment
Raw bad output:
{"label": "billing", "confidence": 1.4, "reason": "Mentions invoice."}
Parsed data: {'label': 'billing', 'confidence': 1.4, 'reason': 'Mentions invoice.'}
Valid: False
Message: Confidence must be between 0 and 1.
Raw bad output:
{"label": "bug", "confidence": 0.7}
Parsed data: {'label': 'bug', 'confidence': 0.7}
Valid: False
Message: Missing required key: reason


### Task 3 observations

The structured classifier returned parseable JSON for all 8 tickets. Each response passed validation because it included a valid label, a confidence value between 0 and 1, and a non-empty reason.

Adding label definitions improved the results. For example, the duplicate charge ticket was correctly classified as `billing`, and the `500 error` ticket was correctly classified as `bug`.

However, valid JSON does not guarantee a perfect classification. The invoice-copy ticket was returned as `other`, even though the label definitions say invoices belong under `billing`. This shows that output validation checks the structure and allowed values, but it does not fully verify semantic correctness.

The bad-output tests were handled gracefully. Invalid JSON, unsupported labels, out-of-range confidence values, and missing fields were all rejected with clear validation messages.

### Local vs hosted comparison

Gemini was not available for live structured-output testing because the API returned a `429 RESOURCE_EXHAUSTED` quota error. Because of that, I completed the lab using the local Ollama `llama3.2:3b` fallback, as allowed in the lab instructions.

The local Ollama model produced valid JSON for all 8 tickets when I used a strict JSON prompt and `response_format={"type": "json_object"}`. However, one result was semantically wrong: the invoice-copy ticket was labeled as `other` instead of `billing`.

This shows the difference between format reliability and classification reliability. The model can return valid JSON that passes validation, but the predicted label can still be wrong. For production use, I would combine structured output validation with better examples, clearer label definitions, and possibly human review for low-confidence or ambiguous cases.